# Cats & Dogs

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
warnings.filterwarnings('ignore')

In [ ]:
# Cargar los data sets de entrenamiento y prueba
file_path = '/kaggle/input/microsoft-catsvsdogs-dataset/PetImages' 

In [ ]:
# Función para verificar imágenes no válidas
def verify_images(directory):
    valid_images = []
    labels = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            try:
                img = Image.open(os.path.join(root, file))
                img.verify()  # Verifica que la imagen no esté corrupta
                valid_images.append(os.path.join(root, file))
                if "Cat" in root:
                    labels.append('cat')
                else:
                    labels.append('dog')
            except (IOError, SyntaxError):
                print(f"Corrupt image detected: {os.path.join(root, file)}")
    return valid_images, labels

# Verificar imágenes antes de procesar
valid_images, labels = verify_images(file_path)

# Crear un DataFrame con las imágenes válidas y sus etiquetas
valid_images_df = pd.DataFrame({'file_path': valid_images, 'label': labels})

# Crear una función generadora personalizada
def custom_image_data_generator(valid_images, batch_size, target_size, subset):
    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.3)
    generator = datagen.flow_from_dataframe(
        dataframe=valid_images,
        x_col='file_path',
        y_col='label',
        target_size=target_size,
        batch_size=batch_size,
        class_mode='binary',
        subset=subset
    )
    return generator

# Configuración del generador de datos de imagen
target_size = (64, 64)
batch_size = 32

train_data = custom_image_data_generator(valid_images_df, batch_size, target_size, 'training')
validation_data = custom_image_data_generator(valid_images_df, batch_size, target_size, 'validation')

# Crear el modelo secuencial
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(train_data, validation_data=validation_data, epochs=10)

# Evaluar el modelo
loss, accuracy = model.evaluate(validation_data)
print(f'Loss: {loss}, Accuracy: {accuracy}')

# Graficar la precisión del entrenamiento y validación
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

# Graficar la pérdida del entrenamiento y validación
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.show()